# 2. Preprocess

In [ ]:
import numpy as np
import pandas as pd
from helpers.pathlib import PATH_DATA, get_path_site_checkpoint

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow
from ata_pipeline1.preprocessors import (
    AddFieldDeviceIsMobile,
    AddFieldEventParentId,
    AddFieldFormSubmitIsNewsletter,
    AddFieldMaxScrollDepth,
    AddFieldsPageType,
    AggregatePageActivities,
    ConvertFieldTypes,
    SortFieldTimestamp,
)
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

In [ ]:
# Read data
df = pd.read_csv(PATH_DATA / "221103-221214.csv")

In [ ]:
df.shape

## Convert types

In [ ]:
df = ConvertFieldTypes()(df)

## Add max scroll depths

In [ ]:
df = AddFieldMaxScrollDepth()(df)

## Sort by ascending timestamp

In [ ]:
df = SortFieldTimestamp()(df)

## Check if device is mobile

In [ ]:
df = AddFieldDeviceIsMobile()(df)

## Get individual site DataFrames

In [ ]:
df_afla = df.query(f"site_name == '{AFRO_LA.name}'")
df_dfp = df.query(f"site_name == '{DALLAS_FREE_PRESS.name}'")
df_ov = df.query(f"site_name == '{OPEN_VALLEJO.name}'")
df_19 = df.query(f"site_name == '{THE_19TH.name}'")

In [ ]:
# CHECKPOINT 1

df_afla.to_pickle(get_path_site_checkpoint(1, AFRO_LA.name))
df_dfp.to_pickle(get_path_site_checkpoint(1, DALLAS_FREE_PRESS.name))
df_ov.to_pickle(get_path_site_checkpoint(1, OPEN_VALLEJO.name))
df_19.to_pickle(get_path_site_checkpoint(1, THE_19TH.name))

## Add newsletter-signup validation

In [ ]:
# Read data from CHECKPOINT 1
df_afla = pd.read_pickle(get_path_site_checkpoint(1, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(1, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(1, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(1, THE_19TH.name))

In [ ]:
df_afla = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=AFRO_LA.newsletter_signup_validator)(df_afla)
df_dfp = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=DALLAS_FREE_PRESS.newsletter_signup_validator)(
    df_dfp
)
df_ov = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=OPEN_VALLEJO.newsletter_signup_validator)(df_ov)
df_19 = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=THE_19TH.newsletter_signup_validator)(df_19)

## Add parent-event column

In [ ]:
assert df_afla.derived_tstamp.is_monotonic_increasing
assert df_dfp.derived_tstamp.is_monotonic_increasing
assert df_ov.derived_tstamp.is_monotonic_increasing
assert df_19.derived_tstamp.is_monotonic_increasing

In [ ]:
# HOW TO COME UP WITH PING_INTERVAL_SECONDS AND ITS EPSILON

# import numpy as np
# from scipy.stats import median_abs_deviation

# print(df_afla.groupby(["domain_userid", "domain_sessionidx", "page_urlpath"], group_keys=True)["derived_tstamp"]
#     .apply(lambda x: x.sort_values().apply(lambda ts: ts.value / 1e9).rolling(2).apply(lambda w: w.iloc[1] - w.iloc[0] if w.shape[0] > 1 else np.nan))
#     .dropna()
#     .aggregate([np.median, median_abs_deviation]))

# print(df_ov.groupby(["domain_userid", "domain_sessionidx", "page_urlpath"], group_keys=True)["derived_tstamp"]
#     .apply(lambda x: x.sort_values().apply(lambda ts: ts.value / 1e9).rolling(2).apply(lambda w: w.iloc[1] - w.iloc[0] if w.shape[0] > 1 else np.nan))
#     .dropna()
#     .aggregate([np.median, median_abs_deviation]))

# print(df_dfp.groupby(["domain_userid", "domain_sessionidx", "page_urlpath"], group_keys=True)["derived_tstamp"]
#     .apply(lambda x: x.sort_values().apply(lambda ts: ts.value / 1e9).rolling(2).apply(lambda w: w.iloc[1] - w.iloc[0] if w.shape[0] > 1 else np.nan))
#     .dropna()
#     .aggregate([np.median, median_abs_deviation]))

# print(df_19.groupby(["domain_userid", "domain_sessionidx", "page_urlpath"], group_keys=True)["derived_tstamp"]
#     .apply(lambda x: x.sort_values().apply(lambda ts: ts.value / 1e9).rolling(2).apply(lambda w: w.iloc[1] - w.iloc[0] if w.shape[0] > 1 else np.nan))
#     .dropna()
#     .aggregate([np.median, median_abs_deviation]))

In [ ]:
# PING_INTERVAL_SECONDS = 10
# PING_INTERVAL_EPSILON_SECONDS = 1  # Ping interval error/noise

In [ ]:
preprocessor_add_field_event_parent_id = AddFieldEventParentId()

In [ ]:
df_afla = preprocessor_add_field_event_parent_id(df_afla)
df_dfp = preprocessor_add_field_event_parent_id(df_dfp)
df_ov = preprocessor_add_field_event_parent_id(df_ov)
df_19 = preprocessor_add_field_event_parent_id(df_19)

In [ ]:
assert df_afla.shape[0] == df.query(f"site_name == '{AFRO_LA.name}'").shape[0]
assert df_dfp.shape[0] == df.query(f"site_name == '{DALLAS_FREE_PRESS.name}'").shape[0]
assert df_ov.shape[0] == df.query(f"site_name == '{OPEN_VALLEJO.name}'").shape[0]
assert df_19.shape[0] == df.query(f"site_name == '{THE_19TH.name}'").shape[0]

In [ ]:
assert df_afla.derived_tstamp.is_monotonic_increasing
assert df_dfp.derived_tstamp.is_monotonic_increasing
assert df_ov.derived_tstamp.is_monotonic_increasing
assert df_19.derived_tstamp.is_monotonic_increasing

In [ ]:
def assert_first_ts_is_also_min_ts(df):
    assert (
        df.groupby(FieldNew.EVENT_PARENT_ID)[FieldSnowplow.DERIVED_TSTAMP].first()
        == df.groupby(FieldNew.EVENT_PARENT_ID)[FieldSnowplow.DERIVED_TSTAMP].min()
    ).all()


assert_first_ts_is_also_min_ts(df_afla)
assert_first_ts_is_also_min_ts(df_dfp)
assert_first_ts_is_also_min_ts(df_ov)
assert_first_ts_is_also_min_ts(df_19)

In [ ]:
# CHECKPOINT 2

df_afla.to_pickle(get_path_site_checkpoint(2, AFRO_LA.name))
df_dfp.to_pickle(get_path_site_checkpoint(2, DALLAS_FREE_PRESS.name))
df_ov.to_pickle(get_path_site_checkpoint(2, OPEN_VALLEJO.name))
df_19.to_pickle(get_path_site_checkpoint(2, THE_19TH.name))

## Aggregate page activities

In [ ]:
# Read data from CHECKPOINT 2
df_afla = pd.read_pickle(get_path_site_checkpoint(2, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(2, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(2, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(2, THE_19TH.name))

In [ ]:
assert df_afla.derived_tstamp.is_monotonic_increasing
assert df_dfp.derived_tstamp.is_monotonic_increasing
assert df_ov.derived_tstamp.is_monotonic_increasing
assert df_19.derived_tstamp.is_monotonic_increasing

In [ ]:
preprocessor_aggregate_page_activities = AggregatePageActivities(
    agg_funcs={
        FieldSnowplow.DERIVED_TSTAMP: (FieldSnowplow.DERIVED_TSTAMP, "first"),
        FieldSnowplow.DOC_HEIGHT: (FieldSnowplow.DOC_HEIGHT, "mean"),
        FieldSnowplow.DOMAIN_SESSIONIDX: (FieldSnowplow.DOMAIN_SESSIONIDX, "first"),
        FieldSnowplow.DOMAIN_USERID: (FieldSnowplow.DOMAIN_USERID, "first"),
        FieldSnowplow.EVENT_ID: (FieldSnowplow.EVENT_ID, "first"),
        FieldSnowplow.EVENT_NAME: (FieldSnowplow.EVENT_NAME, "first"),
        FieldSnowplow.PAGE_URLPATH: (FieldSnowplow.PAGE_URLPATH, "first"),
        FieldNew.DVCE_IS_MOBILE: (FieldNew.DVCE_IS_MOBILE, lambda x: x.mean() > 0.5),
        FieldNew.FORM_SUBMIT_IS_NEWSLETTER: (FieldNew.FORM_SUBMIT_IS_NEWSLETTER, lambda x: x.any()),
        FieldNew.SCROLL_DEPTH_MAX: (FieldNew.SCROLL_DEPTH_MAX, "max"),
        FieldNew.SITE_NAME: (FieldNew.SITE_NAME, "first"),
        FieldNew.DWELL_SECS: (FieldSnowplow.DERIVED_TSTAMP, lambda x: (x.max() - x.min()).total_seconds()),
    },
)

In [ ]:
df_afla_agg = preprocessor_aggregate_page_activities(df_afla)
df_dfp_agg = preprocessor_aggregate_page_activities(df_dfp)
df_ov_agg = preprocessor_aggregate_page_activities(df_ov)
df_19_agg = preprocessor_aggregate_page_activities(df_19)

In [ ]:
def assert_parent_is_self(df):
    assert df.reset_index()[FieldNew.EVENT_PARENT_ID].tolist() == df[FieldSnowplow.EVENT_ID].tolist()


assert_parent_is_self(df_afla_agg)
assert_parent_is_self(df_dfp_agg)
assert_parent_is_self(df_ov_agg)
assert_parent_is_self(df_19_agg)

### Dwell-time gut checks

X-axis unit is in seconds.

In [ ]:
# AfroLA
print(df_afla_agg.dwell_secs.describe())
# df_afla_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# DFP
print(df_dfp_agg.dwell_secs.describe())
# df_dfp_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# OpenVallejo
print(df_ov_agg.dwell_secs.describe())
# df_ov_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# The 19th*
print(df_19_agg.dwell_secs.describe())
# df_19_agg.dwell_secs.plot.box(vert=False)

In [ ]:
# AfroLA: Dwell times among visits where a newsletter form submission happens
df_afla_agg.query(f"{FieldNew.FORM_SUBMIT_IS_NEWSLETTER} == True").dwell_secs.describe()

In [ ]:
# DFP: Dwell times among visits where a newsletter form submission happens
df_dfp_agg.query(f"{FieldNew.FORM_SUBMIT_IS_NEWSLETTER} == True").dwell_secs.describe()

In [ ]:
# OpenVallejo: Dwell times among visits where a newsletter form submission happens
df_ov_agg.query(f"{FieldNew.FORM_SUBMIT_IS_NEWSLETTER} == True").dwell_secs.describe()

In [ ]:
# The 19th*: Dwell times among visits where a newsletter form submission happens
df_19_agg.query(f"{FieldNew.FORM_SUBMIT_IS_NEWSLETTER} == True").dwell_secs.describe(percentiles=np.arange(0, 1, 0.1))

In [ ]:
# CHECKPOINT 3

df_afla_agg.to_pickle(get_path_site_checkpoint(3, AFRO_LA.name))
df_dfp_agg.to_pickle(get_path_site_checkpoint(3, DALLAS_FREE_PRESS.name))
df_ov_agg.to_pickle(get_path_site_checkpoint(3, OPEN_VALLEJO.name))
df_19_agg.to_pickle(get_path_site_checkpoint(3, THE_19TH.name))

## Classify page type

In [ ]:
# Read data from CHECKPOINT 3
df_afla_agg = pd.read_pickle(get_path_site_checkpoint(3, AFRO_LA.name))
df_dfp_agg = pd.read_pickle(get_path_site_checkpoint(3, DALLAS_FREE_PRESS.name))
df_ov_agg = pd.read_pickle(get_path_site_checkpoint(3, OPEN_VALLEJO.name))
df_19_agg = pd.read_pickle(get_path_site_checkpoint(3, THE_19TH.name))

In [ ]:
df_afla_agg = AddFieldsPageType(site_page_type_classifier=AFRO_LA.page_type_classifier)(df_afla_agg)

In [ ]:
df_dfp_agg = AddFieldsPageType(site_page_type_classifier=DALLAS_FREE_PRESS.page_type_classifier)(df_dfp_agg)

In [ ]:
df_ov_agg = AddFieldsPageType(site_page_type_classifier=OPEN_VALLEJO.page_type_classifier)(df_ov_agg)

In [ ]:
# This takes about 4 minutes and 2 GB of memory to run.
# If that's a problem maybe look into batching (as part of the Preprocessing base class so it's reusable)?
df_19_agg = AddFieldsPageType(site_page_type_classifier=THE_19TH.page_type_classifier)(df_19_agg)

In [ ]:
# CHECKPOINT 4

df_afla_agg.to_pickle(get_path_site_checkpoint(4, AFRO_LA.name))
df_dfp_agg.to_pickle(get_path_site_checkpoint(4, DALLAS_FREE_PRESS.name))
df_ov_agg.to_pickle(get_path_site_checkpoint(4, OPEN_VALLEJO.name))
df_19_agg.to_pickle(get_path_site_checkpoint(4, THE_19TH.name))